# 03 — Tomography & Fidelity on Real Hardware

Counts hand you probabilities — one number per outcome. But the device prepared a *whole* state, with phases and coherences that a single histogram cannot see. To recover all of it you measure in three complementary bases and reconstruct the state, then ask the honest question: how far did it fall from the ideal you asked for?

**Prerequisites:** `01/15_fidelity_and_trace_distance`, `01/16_tomography_of_a_noisy_qubit`, `05/01_using_a_real_quantum_computer`.

**What you'll be able to do**
- Run single-qubit Pauli tomography through the runtime primitives (`SamplerV2`), one circuit per basis.
- Reconstruct the device's density matrix `rho_hat` from the counts with `density_from_counts`.
- Read a purity below 1 and a fidelity below 1, and name what each one measures.
- Set the *ideal* `rho` you asked for beside the *device* `rho_hat` you got back, in the ideal-versus-device table of this module — pure giving way to mixed.

This is the primitive-path successor to `01/16`, where you ran tomography with a local `backend.run`. The mathematics is identical; what changes is that the same code now reaches a real QPU through `SamplerV2`. You already know how to turn counts into a state — here you turn a *device's* counts into the state it actually prepared, and then you measure the gap. Reconstructing a real quantum state from nothing but click statistics is a genuine milestone, so let us go and do it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2
from qot_course.hardware.records import save_run_record
from qot_course.hardware.runtime import select_backend
from qot_course.hardware.tomography import single_qubit_tomography_circuits, density_from_counts
from qot_course.quantum.density import purity, fidelity, density_matrix
from qot_course.quantum.states import KET_PLUS

np.random.seed(42)
viz.use_course_style()

In [ ]:
USE_HARDWARE = False  # flip to True after you've set up credentials (see 05/01)
backend, label, is_real = select_backend(use_hardware=USE_HARDWARE)
print(f"Running on: {label}  (real hardware = {is_real})")
if not is_real:
    backend.set_options(seed_simulator=42)  # fixed seed: the offline numbers quoted below are reproducible

## 1 · Measuring in three bases

A single qubit's state is fixed completely by three numbers: the expectation values $\langle X \rangle$, $\langle Y \rangle$, and $\langle Z \rangle$, which together are its Bloch vector. A measurement only ever reads the $Z$ axis, so to learn the other two we rotate them onto $Z$ first: one circuit measures $\langle Z \rangle$ directly, another rotates $X$ onto $Z$ before measuring, the third rotates $Y$. Three circuits, three Bloch components, one reconstructed state. The helper `single_qubit_tomography_circuits` builds all three for us — it takes the state-preparation circuit and returns a dictionary keyed `"X"`, `"Y"`, `"Z"`.

In [ ]:
prep = QuantumCircuit(1)
prep.h(0)                                   # the |+> we try to prepare
circuits = single_qubit_tomography_circuits(prep)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
settings = list(circuits)                   # ["X", "Y", "Z"]
isa_list = [pm.run(circuits[s]) for s in settings]
SHOTS = 4096
job = SamplerV2(mode=backend).run(isa_list, shots=SHOTS)
res = job.result()
counts_by_setting = {s: res[i].data.c.get_counts() for i, s in enumerate(settings)}
save_run_record(counts_by_setting, notebook="05_03", tag="tomography_plus", device=backend.name, shots=SHOTS,
                job_id=job.job_id() if is_real else None, is_real=is_real)
print("physical qubit:", isa_list[0].layout.final_index_layout()[0])

rho_hat = density_from_counts(counts_by_setting, n_qubits=1)
ideal = density_matrix(KET_PLUS)

# Before the projection: the linear-inversion estimate has eigenvalues (1 +/- |r|)/2, r the Bloch vector.
bloch = np.array([(counts_by_setting[s].get("0", 0) - counts_by_setting[s].get("1", 0)) / SHOTS for s in ("X", "Y", "Z")])
eig_lin = 0.5 * (1 + np.array([-1.0, 1.0]) * np.linalg.norm(bloch))
print("Bloch vector (X, Y, Z):", np.round(bloch, 4), " length:", round(float(np.linalg.norm(bloch)), 4))
print("eigenvalues before projection:", np.round(eig_lin, 4), " clipped to zero:", int((eig_lin < 0).sum()))

# Statistical error bars: bootstrap-resample each setting's counts (multinomial), as in 05/06.
rng = np.random.default_rng(0)
pur_boot, fid_boot, clipped_boot = [], [], 0
for _ in range(200):
    resampled = {}
    for s, c in counts_by_setting.items():
        keys = list(c)
        draws = rng.multinomial(SHOTS, np.array([c[k] for k in keys], dtype=float) / SHOTS)
        resampled[s] = dict(zip(keys, draws.tolist()))
    r_vec = [(resampled[s].get("0", 0) - resampled[s].get("1", 0)) / SHOTS for s in ("X", "Y", "Z")]
    clipped_boot += int(np.linalg.norm(r_vec) > 1)
    rho_b = density_from_counts(resampled, n_qubits=1)
    pur_boot.append(purity(rho_b)); fid_boot.append(fidelity(rho_b, ideal))
print(f"purity(rho_hat)       : {purity(rho_hat):.3f} +/- {np.std(pur_boot):.3f} (stat)   (< 1  =>  mixed)")
print(f"fidelity(rho_hat, |+>): {fidelity(rho_hat, ideal):.3f} +/- {np.std(fid_boot):.3f} (stat)   (< 1  =>  imperfect prep)")
print(f"bootstrap replicas that needed the eigenvalue clip: {clipped_boot} of 200")
print("rho_hat =\n", np.round(rho_hat, 3))

In [ ]:
viz.plot_density_matrix(rho_hat, title="Tomography of |+> on the device", basis_labels=["|0>", "|1>"])
plt.show()

**Read the figure.** The reconstructed `rho_hat` has off-diagonal coherences of $0.489$ (printed above), a little below the $0.5$ of the ideal $|+\rangle\langle+|$, and its diagonal leans slightly, $0.487$ against $0.513$. Those shrunken corners, together with a purity below 1, are *imperfection* captured in a single picture: decoherence (the energy relaxation and dephasing that `01/21` timed with $T_1$ and $T_2$), gate and read-out error, and finite-shot bias. On this run the eigenvalue clip did nothing. The linear-inversion estimate had eigenvalues $0.0113$ and $0.9887$, both positive, and none of the 200 bootstrap replicas needed the clip either, so the projection added no bias here. Finite shots still move the estimate: the bootstrap puts a statistical error bar of $0.003$ on the purity and $0.002$ on the fidelity. Raising the shot count shrinks that part, and what remains is the physical gap. Offline this reflects the modeled FakeMarrakesh noise; flip `USE_HARDWARE=True` and the very same plot shows the literal device's state.

### A real run on IBM hardware

On **`ibm_marrakesh`** (2026-05-30), single-qubit tomography of `|+>` reconstructed a state with **purity 0.974** and **fidelity 0.986** to the ideal, both below 1. That run was one pass on a physical qubit that was not recorded, with no error bar, and its job identifier and raw counts were not kept, so the values cannot be recomputed from this repository. The offline twin of the same device gives $0.978 \pm 0.003$ and $0.989 \pm 0.002$ above, and the next cell shows that the real values sit between one and two statistical error bars below them: as close as one unrepeated run can be compared. Set `USE_HARDWARE = True` to reproduce; the cell above now prints the physical qubit and the error bars, and saves the counts.

In [ ]:
# The real run quoted above, set against the offline twin's bootstrap error bars.
REAL_PURITY, REAL_FIDELITY = 0.974, 0.986   # ibm_marrakesh, 2026-05-30; raw counts not kept
print(f"purity  : real - offline = {REAL_PURITY - purity(rho_hat):+.3f} = {(REAL_PURITY - purity(rho_hat)) / np.std(pur_boot):+.1f} error bars")
print(f"fidelity: real - offline = {REAL_FIDELITY - fidelity(rho_hat, ideal):+.3f} = {(REAL_FIDELITY - fidelity(rho_hat, ideal)) / np.std(fid_boot):+.1f} error bars")

## 2 · Ideal versus device

Module 05 keeps a small table of its own, which sets each ideal object beside what a real device hands back. It is not part of the classical ↔ quantum dictionary of Modules 01 to 04. Tomography gives its first rows, which close module 01's density-matrix arc:

| Ideal object | What a real device returns |
|---|---|
| pure state $|\psi\rangle$ | mixed state `rho_hat` (purity < 1) |
| exact `rho` | tomographic reconstruction from finite shots |
| fidelity 1 with the target | fidelity < 1 — the decoherence gap |

Module 01 introduced the density matrix as the language for imperfect, mixed states — exactly the situation a real qubit lives in. Here you watched that language earn its place: the device did not return a pure `|+>`, it returned a mixed `rho_hat`, and fidelity gave you a single number for how far the two stand apart.

## Your turn

- **Tomograph `|0>` instead.** Set `prep = QuantumCircuit(1)` with no gate (or `prep.id(0)`) and re-run. Preparing `|0>` needs no state-prep gate, so it escapes the Hadamard's gate error. Does the purity rise, or does the read-out error, which hits every state, outweigh the gate error of one Hadamard? Run a few seeds before you decide, and use the calibration of `05/02` to see how the read-out treats `0` and `1` on this qubit.
- **Flip `USE_HARDWARE = True`** (with credentials set per `05/01`) and re-run. The purity and fidelity gaps you read are now the real chip's, not a model's.
- **Watch the gap grow.** Compute `1 - fidelity` and track it as you add gates before tomography — append an `h` then an `id`, or a short string of gates (note: two H gates compose to identity, so the *target stays the same* while the accumulated gate error grows) — and see the infidelity climb as the circuit lengthens. That growing `1 - fidelity` is a preview of the Bures sweep you build in `05/05`.

## What you built

- A full single-qubit tomography run on the primitive path: three Pauli-basis circuits transpiled, sampled with `SamplerV2`, and reconstructed into a density matrix `rho_hat` with `density_from_counts`.
- Two scores for the device's state, each with a bootstrap error bar: a purity below 1 that exposes it as mixed, and a fidelity below 1 that measures the gap to the ideal `|+>`.
- The first rows of the ideal-versus-device table, tying the ideal `rho` you asked for to the device `rho_hat` you got — the decoherence gap made quantitative.

## What's next

Next you lift these ideas to two qubits and measure **quantum mutual information** on real hardware — the information one qubit's tomography reveals about another, and the first information-geometric quantity of the module to meet a live device.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 8.4 (state tomography) and ch. 9 (fidelity and distance measures), Cambridge University Press (2010).
- D. F. V. James, P. G. Kwiat, W. J. Munro & A. G. White, "Measurement of qubits," *Phys. Rev. A* **64**, 052312 (2001), doi:10.1103/PhysRevA.64.052312.

**Previous:** `notebooks/05_RealQuantumHardware/02_born_rule_on_real_hardware.ipynb` · **Next:** `notebooks/05_RealQuantumHardware/04_quantum_mutual_information_on_real_hardware.ipynb`